In [ ]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import os
import joblib
from copy import deepcopy
import torch
import chromadb
import gc
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass
import matplotlib.pyplot as plt

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics
from src.utils.inference_metrics import compute_predictive_entropy, get_timportance_info

os.environ["TRANSFORMERS_VERBOSITY"] = "error"

CONTEXTS_DATASET_PATH = "../../../../data/squadv2/contexts.csv"
QA_DATASET_PATH = "../../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../../models/Qwen/Qwen2.5-7B-Instruct" # "Undi95/Meta-Llama-3-8B-Instruct-hf" / "../../../../models/Qwen/Qwen2.5-7B-Instruct"

In [2]:
!pip install evaluate torchmetrics langchain_huggingface levenshtein sentence_transformers chromadb tqdm torch transformers nltk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.6/962.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 13.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 11.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 12.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
PARAMS = {
    'version': "3.1.2.attn",
    'num_samples': 2000,
    'num_contexts': 5,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. If there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024, 'do_sample': False, 'num_beams': 1},
    'stub_answer': "I do not have an answer to your question",
    'calculate_entropy': True,
    'calculate_timportance(attention)': True,
    'timportnace_hyperp': {'layers': [0,1,13,27], 'mean_by': ['columns', 'rows']},
    'revert': False,
    'centered': True
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
TIMPORTANCE_SAVE_NAME = 'timportance'
LOGS_SAVE_DIR = './logs_v2'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

### Подключение к агенту

In [3]:
agent = CustomAgent(PARAMS['model'], output_logits=PARAMS['calculate_entropy'], use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], 
                        gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The question of what is "wrong" with humanity is complex and multifaceted, as it can be interpreted in various ways. Here are some common perspectives:

1. **Inequality and Discrimination**:**: Human societies often struggle with issues of inequality, discrimination based on race, gender, sexuality, and other factors, which can lead to social unrest and injustice.

2. **Environmental Degradation**:**: Many people believe that humanity is failing to adequately address environmental issues such as climate change, deforestation, pollution, and loss of biodiversity.

3. **Conflict and War**:**: Despite on various grounds, humanity continues to engage in conflicts and wars, which result in significant loss of life and suffering.

4. **Moral and Ethical Dilemmas**:**: There are ongoing debates about moral and ethical issues such as human rights, animal welfare, and the treatment of marginalized groups.

5. **Technological and Social Challenges**:**: The rapid pace of technological change can

In [4]:
agent.model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

### Формируем список контекстов для каждого запроса со скорами

In [5]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [6]:
contexts_df = pd.read_csv(CONTEXTS_DATASET_PATH)

In [7]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_rel_id = int(dataset_df['relevant_context_id'][i])
    cur_list_ids = [(-1, cur_rel_id)]

    while len(cur_list_ids) != PARAMS['num_contexts']:
        unrel_context_id = random.randint(0, contexts_df.shape[0]-1)

        prep_cntx = (-1, unrel_context_id)
        if unrel_context_id != cur_rel_id:
            cur_list_ids.append(prep_cntx)

    # shuffling strategy
    if PARAMS['revert']:
        cur_list_ids = cur_list_ids[::-1]
    elif PARAMS['centered']:
        cur_list_ids.pop(0)
        cur_list_ids.insert(len(cur_list_ids)//2, (-1, cur_rel_id))
    
    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 2000/2000 [00:00<00:00, 192500.81it/s]


In [8]:
print(CONTEXTS_LIST_IDS[0])

[(-1, 3648), (-1, 819), (-1, 0), (-1, 9012), (-1, 8024)]


### Готовим промпт

In [9]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    docs = [contexts_df['context'][CONTEXTS_LIST_IDS[i][j][1]] for j in range(len(CONTEXTS_LIST_IDS[i]))]
    documents_list = [PARAMS['item_format'].format(document=doc.strip()) for doc in docs]
    
    documents_list = '\n'.join(documents_list)
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 2000/2000 [00:00<00:00, 70986.43it/s]


In [10]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [11]:
del contexts_df
gc.collect()

66

In [12]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. If there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- Gombeenism refers to an individual who is dishonest and corrupt for the purpose of personal gain, more often through monetary, while, parochialism which is also known as parish pump politics relates to placing local or vanity projects ahead of the national interest.For instance in Irish politics, populist left wing political parties will often apply these terms to mainstream establisment political parties and will cite the many cases of Corruption in Ireland, such as the Irish Banking crisis, which found evidence of bribery,

### Генерируем ответы на вопросы

In [13]:
generate_answers, calc_metrics = [], []
timportance_info = dict()
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info, inputs = agent.generate(
        user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'],
        gen_strategy=PARAMS['gen_strat'])

    cur_metrics = dict()
    cur_metrics['input_tokens'] = inputs['input_ids'].shape[1]
    cur_metrics['gen_tokens'] = meta_info['sequences'].shape[1] - cur_metrics['input_tokens']
    
    if PARAMS['calculate_entropy']:
        logits = torch.cat(meta_info['logits'], 0).cpu().detach()
        entropy = compute_predictive_entropy(logits)
        cur_metrics['predictive_entropy'] = float(entropy)
    
    if PARAMS['calculate_timportance(attention)']:
        agent.output_attentions = True
        tmp_assistant_prompt = pred_answer
        _, meta_info, _ = agent.generate(
            user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], 
            gen_strategy={'max_new_tokens': 1, 'do_sample': False, 'num_beams': 1}, 
            assistant_prompt=tmp_assistant_prompt)
        timportance_info[i] = get_timportance_info(
            meta_info['attentions'][0], layer_ids = PARAMS['timportnace_hyperp']['layers'], 
            mean_attn = PARAMS['timportnace_hyperp']['mean_by'])
        agent.output_attentions = False
        
    calc_metrics.append(cur_metrics)
    generate_answers.append(pred_answer)
    
    # logits = torch.cat(meta_info['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}\nMETRICS: {cur_metrics}")
e_time = time()

  0%|          | 1/2000 [00:02<1:23:22,  2.50s/it]


[0]: 
GEN: Beyoncé started becoming popular in the late 1990s as the lead singer of Destiny's Child, but her solo career gained prominence with the release of "Dangerously in Love" in 2003.
GOLD: in the late 1990s
METRICS: {'input_tokens': 1115, 'gen_tokens': 48, 'predictive_entropy': 8.875941276550293}


  5%|▌         | 101/2000 [02:22<43:54,  1.39s/it]


[100]: 
GEN: eleven
GOLD: eleven
METRICS: {'input_tokens': 1176, 'gen_tokens': 3, 'predictive_entropy': 0.4365878105163574}


 10%|█         | 201/2000 [05:16<43:56,  1.47s/it]  

...

 90%|█████████ | 1801/2000 [42:00<04:05,  1.23s/it]


[1800]: 
GEN: Kane Kramer called the device for which he wanted a patent the IXI.
GOLD: IXI
METRICS: {'input_tokens': 904, 'gen_tokens': 17, 'predictive_entropy': 0.522372841835022}


 95%|█████████▌| 1901/2000 [44:07<02:12,  1.34s/it]


[1900]: 
GEN: September 12, 2006
GOLD: September 12, 2006
METRICS: {'input_tokens': 994, 'gen_tokens': 11, 'predictive_entropy': 0.777260959148407}


100%|██████████| 2000/2000 [46:08<00:00,  1.38s/it]


In [14]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), int(item[1])) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {
        'gen_answer': str(generate_answers[i]), 
        'metainfo': calc_metrics[i], 
        'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

if PARAMS['calculate_timportance(attention)']:
    joblib.dump(timportance_info, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/{TIMPORTANCE_SAVE_NAME}")

### Оцениваем качество

In [19]:
LOADING_VERSION = "3.1.2.attn"

In [20]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [21]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [22]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [23]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [24]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

  0%|          | 0/2000 [00:00<?, ?it/s]/opt/conda/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [04:50<00:00,  6.88it/s, BLEU2=0.00491, BLEU1=0.0133, ExactMatch=0.00552, METEOR=0.0161, BertScore=nan, Levenshtain=49.1, ROUGEL=0.0162] 
The following layers were not sharded: encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.output.dense.bias, embeddings.word_embeddings.weight, embeddings.LayerNorm.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.key.weight, embeddings.position_embeddings.weight, e

In [25]:
LOADING_VERSION

'3.1.2.attn'

In [26]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))